In [2]:
# !pip install git+https://github.com/facebookresearch/segment-anything.git
# !pip install opencv-python pycocotools matplotlib onnxruntime onnx

In [ ]:
import os
import glob
import time
import gc
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import timedelta
from PIL import Image
from ultralytics import YOLO
import torch
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window
from shapely.geometry import Polygon
from shapely.ops import unary_union
from sklearn.cluster import DBSCAN
from pyproj import Transformer
import cv2
import pandas as pd
import numpy as np
import json
from shapely.geometry import Polygon
from sklearn.cluster import DBSCAN
from shapely.geometry import Polygon
import numpy as np
import torch
from segment_anything import sam_model_registry, SamPredictor

from segment_anything import sam_model_registry, SamPredictor

sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"
device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device)
predictor = SamPredictor(sam)

In [10]:
def run_sam_on_min_rects(crop_img, crop_transform, yolo_object_infos, offset_xy, predictor):
    """
    For each YOLO object: get min bounding rectangle of mask, crop region, run SAM, save results.
    Args:
        crop_img: (C, H, W) numpy array of cropped window.
        crop_transform: rasterio transform for the cropped region.
        yolo_object_infos: List of object_info dicts (from YOLO).
        offset_xy: (left, top) offset of the crop in global coords.
        predictor: a loaded SamPredictor object.
    Returns:
        List of dicts, same structure as object_info, but from SAM output.
    """
    sam_objects = []
    # Move (C, H, W) -> (H, W, C) and to uint8
    img_rgb = np.moveaxis(crop_img[:3], 0, -1)
    img_rgb = np.clip((img_rgb / img_rgb.max()) * 255, 0, 255).astype(np.uint8)
    predictor.set_image(img_rgb)

    for obj in yolo_object_infos:
        # Get YOLO mask points in local (cropped) coordinates
        mask_pts = np.array(json.loads(obj["mask_points"]), dtype=np.int32)
        # Fit minimum rectangle
        poly = Polygon(mask_pts)
        if not poly.is_valid or poly.is_empty:
            continue
        min_rect = poly.minimum_rotated_rectangle
        min_rect_coords = np.array(list(min_rect.exterior.coords), dtype=np.int32)

        # Center point for SAM prompt (in local crop image)
        center_x = int(min_rect.centroid.x)
        center_y = int(min_rect.centroid.y)
        input_point = np.array([[center_x, center_y]])
        input_label = np.array([1])

        # Run SAM
        masks, scores, logits = predictor.predict(
            point_coords=input_point,
            point_labels=input_label,
            multimask_output=False
        )
        # Only one mask expected if multimask_output=False
        mask = masks[0]  # (H, W) bool array

        # Convert mask to polygons
        mask_points = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
        if len(mask_points) == 0:
            continue
        mask_points = mask_points[0].squeeze()
        if mask_points.ndim != 2 or mask_points.shape[0] < 3:
            continue  # skip invalid

        # Map pixel mask points to GIS coordinates
        mask_gis_coords = []
        for px, py in mask_points:
            try:
                lon, lat = rasterio.transform.xy(crop_transform, py, px)
                mask_gis_coords.append([lon, lat])
            except:
                continue

        # Calculate center in geo coords
        lon_center, lat_center = rasterio.transform.xy(crop_transform, center_y, center_x)
        pixel_count = mask.sum()
        area_geo = pixel_count * (crop_transform.a * abs(crop_transform.e))

        # Store results
        sam_objects.append({
            **obj,  # inherit original YOLO info for this crop
            "sam_mask_points": json.dumps(mask_points.tolist()),
            "sam_mask_points_gis": json.dumps(mask_gis_coords),
            "sam_center_lat": lat_center,
            "sam_center_lon": lon_center,
            "sam_area_geo": area_geo,
            "sam_pixel_count": int(pixel_count)
        })
    return sam_objects



# ======================== Drawing ========================
def draw_annotations_on_image(full_image_array, slices, object_infos, output_path):
    """
    Overlays bounding boxes, masks, and lat/lon centers with:
    - Transparent masks (alpha=0.3)
    - Class/conf labels
    - Red dot + lat/lon text
    - JPEG output
    """
    import cv2
    import numpy as np
    import json
    from rasterio.transform import rowcol

    # Convert rasterio (C,H,W) to OpenCV (H,W,C)
    img = np.moveaxis(full_image_array[:3], 0, -1)
    img = np.clip((img / img.max()) * 255, 0, 255).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    overlay = img.copy()

    # Offset lookup
    slice_offsets = {tuple(slice_data["offset_xy"]): slice_data for slice_data in slices}

    for obj in object_infos:
        tile_name = obj['tile_name']
        offset_xy = (int(obj['global_x'] - obj['x_center']), int(obj['global_y'] - obj['y_center']))

        # Bounding box
        x1_global = int(obj['bbox_x1'] + offset_xy[0])
        y1_global = int(obj['bbox_y1'] + offset_xy[1])
        x2_global = int(obj['bbox_x2'] + offset_xy[0])
        y2_global = int(obj['bbox_y2'] + offset_xy[1])

        conf = obj['confidence']
        cls = obj['class']
        label = f"Cls:{cls} ({conf:.2f})"

        # Draw bounding box
        cv2.rectangle(overlay, (x1_global, y1_global), (x2_global, y2_global), (0, 255, 0), 3)

        # Label text
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 2.5
        thickness = 5
        text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]
        text_origin = (x1_global, max(y1_global - 10, text_size[1]))
        cv2.putText(overlay, label, text_origin, font, font_scale, (0, 0, 0), thickness + 2)
        cv2.putText(overlay, label, text_origin, font, font_scale, (255, 255, 255), thickness)

        # Mask overlay
        try:
            mask_points = np.array(json.loads(obj['mask_points']), dtype=np.int32)
            mask_points_global = mask_points + np.array([offset_xy[0], offset_xy[1]])

            if len(mask_points_global) >= 3:
                mask_img = np.zeros_like(img)
                cv2.fillPoly(mask_img, [mask_points_global], (0, 0, 255))
                cv2.polylines(mask_img, [mask_points_global], isClosed=True, color=(0, 0, 255), thickness=2)
                overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.7, 0)
        except:
            pass

        # Lat/Lon center overlay
        try:
            lat = float(obj['latitude'])
            lon = float(obj['longitude'])

            # Use reverse transform to get row,col (y,x)
            for slice_data in slices:
                transform = slice_data['transform']
                try:
                    row, col = rowcol(transform, lon, lat)
                    pixel_x = col + slice_data["offset_xy"][0]
                    pixel_y = row + slice_data["offset_xy"][1]
                    break
                except:
                    continue

            # Draw center point
            cv2.circle(overlay, (int(pixel_x), int(pixel_y)), 8, (0, 0, 255), -1)

            # Add lat/lon text
            ll_label = f"({lat:.5f}, {lon:.5f})"
            cv2.putText(overlay, (int(pixel_x + 10), int(pixel_y - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 3)
            cv2.putText(overlay, (int(pixel_x + 10), int(pixel_y - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
        except:
            continue

    # Save as JPEG
    output_path_jpg = output_path.replace(".png", ".jpg")
    cv2.imwrite(output_path_jpg, overlay)
    print(f"📸 Annotated image saved to: {output_path_jpg}")

# ======================== Sliding ========================
from rasterio.windows import Window
from rasterio.transform import from_origin

def slide_geotiff(geotiff_path, window_size, stride):
    """
    Splits a GeoTIFF into window_size×window_size tiles with given stride,
    **and** adds extra overlapping tiles on the right/bottom so that no pixel is left out.
    """
    tiles = []
    with rasterio.open(geotiff_path) as src:
        height, width = src.height, src.width

        # 1) Compute all x-starts
        if width <= window_size:
            x_starts = [0]
        else:
            x_starts = list(range(0, width - window_size + 1, stride))
            # leftover on right?
            if x_starts[-1] + window_size < width:
                x_starts.append(width - window_size)

        # 2) Compute all y-starts
        if height <= window_size:
            y_starts = [0]
        else:
            y_starts = list(range(0, height - window_size + 1, stride))
            # leftover on bottom?
            if y_starts[-1] + window_size < height:
                y_starts.append(height - window_size)

        # 3) Build tiles for every (x,y) pair
        for y in y_starts:
            for x in x_starts:
                window = Window(x, y, window_size, window_size)
                slice_img = src.read(window=window)
                new_transform = from_origin(
                    src.transform.c + x * src.transform.a,
                    src.transform.f + y * src.transform.e,
                    src.transform.a,
                    -src.transform.e
                )
                tiles.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "crs": src.crs,
                    "position": (y, x),
                    "offset_xy": (x, y)
                })

    return tiles


# ======================== Per-Tile Suppression ========================
from torchvision.ops import nms
def maximum_likelihood_suppression(results, conf_threshold=0.7, iou_threshold=0.5):
    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return results
    boxes = results.boxes.xyxy
    scores = results.boxes.conf
    keep = scores >= conf_threshold
    if keep.sum() == 0:
        results.boxes.data = results.boxes.data[[]]
        if results.masks and results.masks.data is not None:
            results.masks.data = results.masks.data[[]]
        return results
    keep_indices = nms(boxes[keep], scores[keep], iou_threshold)
    indices_to_keep = torch.nonzero(keep).squeeze(1)[keep_indices]
    results.boxes.data = results.boxes.data[indices_to_keep]
    if results.masks and results.masks.data is not None:
        results.masks.data = results.masks.data[indices_to_keep]
    return results

# ======================== Per-Tile Extraction ========================
def overlay_segmentation_on_array(results, transform, img_shape, tif_image, offset_xy):
    object_data = []
    masks_out = []

    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return object_data, masks_out

    for idx, (mask, box, cls, conf) in enumerate(zip(
        results.masks.xy, results.boxes.xyxy, results.boxes.cls, results.boxes.conf)):

        cls = int(cls.item())
        conf = float(conf.item())
        x1, y1, x2, y2 = map(float, box.tolist())

        mask_pixel_coords = np.array(mask.tolist(), dtype=np.float32)
        mask_gis_coords = []
        for px, py in mask_pixel_coords:
            try:
                lon, lat = rasterio.transform.xy(transform, py, px)
                mask_gis_coords.append([lon, lat])
            except:
                continue

        pixel_count = len(mask_pixel_coords)
        area = pixel_count * (transform.a * abs(transform.e))
        local_cx = (x1 + x2) / 2
        local_cy = (y1 + y2) / 2
        global_x = offset_xy[0] + local_cx
        global_y = offset_xy[1] + local_cy
        lon_center, lat_center = rasterio.transform.xy(transform, local_cy, local_cx)
        lon_x1, lat_y1 = rasterio.transform.xy(transform, y1, x1)
        lon_x2, lat_y2 = rasterio.transform.xy(transform, y2, x2)
        bbox_width_geo = abs(lon_x2 - lon_x1)
        bbox_height_geo = abs(lat_y2 - lat_y1)

        object_data.append({
            "latitude": lat_center,
            "longitude": lon_center,
            "bbox_lat_y1": lat_y1,
            "bbox_lon_x1": lon_x1,
            "bbox_lat_y2": lat_y2,
            "bbox_lon_x2": lon_x2,
            "bbox_width_geo": bbox_width_geo,
            "bbox_height_geo": bbox_height_geo,
            "global_x": global_x,
            "global_y": global_y,
            "x_center": local_cx,
            "y_center": local_cy,
            "bbox_x1": x1,
            "bbox_y1": y1,
            "bbox_x2": x2,
            "bbox_y2": y2,
            "bbox_width": x2 - x1,
            "bbox_height": y2 - y1,
            "area_geo": area,
            "pixel_count": pixel_count,
            "class": cls,
            "confidence": conf,
            "tile_name": tif_image,
            "mask_index": idx,
            "mask_points": json.dumps(mask_pixel_coords.tolist()),
            "mask_points_gis": json.dumps(mask_gis_coords)
        })

        masks_out.append(mask_pixel_coords)

    return object_data, masks_out

#---------------------------------------------centroid calculation
def detect_poultry_farms(object_infos, eps_house=50, min_samples=2, eps_merge=100):
    # Filter out entries missing coordinates
    filtered_infos = [
        o for o in object_infos
        if 'latitude' in o and 'longitude' in o and pd.notnull(o['latitude']) and pd.notnull(o['longitude'])
    ]
    if not filtered_infos:
        return pd.DataFrame(columns=['merged_farm_id', 'longitude', 'latitude', 'num_houses'])
    df = pd.DataFrame(filtered_infos)
    coords = df[['longitude', 'latitude']].values
    dbscan = DBSCAN(eps=eps_house, min_samples=min_samples).fit(coords)
    df['initial_cluster'] = dbscan.labels_
    clustered = df[df['initial_cluster'] != -1].copy()
    if len(clustered) == 0:
        return pd.DataFrame(columns=['merged_farm_id', 'longitude', 'latitude', 'num_houses'])
    centroids = clustered.groupby('initial_cluster')[['longitude', 'latitude']].mean().reset_index()
    centroid_coords = centroids[['longitude', 'latitude']].values
    merge_cluster = DBSCAN(eps=eps_merge, min_samples=1).fit(centroid_coords)
    centroids['merged_farm_id'] = merge_cluster.labels_
    cluster_map = dict(zip(centroids['initial_cluster'], centroids['merged_farm_id']))
    clustered['merged_farm_id'] = clustered['initial_cluster'].map(cluster_map)
    farms = clustered.groupby('merged_farm_id')[['longitude', 'latitude']].mean().reset_index()
    farms['num_houses'] = clustered.groupby('merged_farm_id').size().values
    return farms


def lonlat_to_pixel(transform, lon, lat):
    # Returns (row, col) for rasterio window indexing
    from rasterio.transform import rowcol
    row, col = rowcol(transform, lon, lat)
    return row, col

def center_crop_window(row, col, img_width, img_height, window_size):
    """Returns a rasterio.Window such that the window is centered on (col, row)."""
    half = window_size // 2
    left = max(col - half, 0)
    top = max(row - half, 0)
    width = min(window_size, img_width - left)
    height = min(window_size, img_height - top)
    return Window(left, top, width, height), (left, top)

def run_inference_centroid_targeted(
    folder_path, model_path, final_csv_path, window_size=3000, batch_size=16,
    eps_house=50, min_samples=2, eps_merge=100
):
    model = YOLO(model_path).to('cuda' if torch.cuda.is_available() else 'cpu')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    all_cropped_object_infos = []

    for tif_image in tqdm(tif_images, desc="GeoTIFFs"):
        print(f"Processing {tif_image}")
        # 1. Initial sliding window inference for farm clustering
        initial_object_infos = []
        with rasterio.open(tif_image) as src:
            img_width, img_height = src.width, src.height
            slices = []
            stride = window_size  # full tiling, non-overlapping
            for y in range(0, img_height, stride):
                for x in range(0, img_width, stride):
                    w = min(window_size, img_width - x)
                    h = min(window_size, img_height - y)
                    window = Window(x, y, w, h)
                    img = src.read(window=window)
                    # Ensure 3 channels for YOLO
                    if img.shape[0] > 3:
                        img = img[:3]
                    elif img.shape[0] < 3:
                        img = np.tile(img, (3 // img.shape[0], 1, 1))
                    pil_img = Image.fromarray(np.moveaxis(img, 0, -1))
                    with torch.inference_mode():
                        results = model([pil_img], device=device, batch=1, verbose=False)[0]
                    results = maximum_likelihood_suppression(results)
                    offset_xy = (x, y)
                    object_info, _ = overlay_segmentation_on_array(
                        results, src.window_transform(window), None, os.path.basename(tif_image), offset_xy)
                    initial_object_infos.extend(object_info)

            # 2. Farm clustering
            farm_centroids_df = detect_poultry_farms(initial_object_infos, eps_house, min_samples, eps_merge)
            if farm_centroids_df.empty:
                print("No farms detected in this image.")
                continue

            # 3. For each centroid, crop, run inference, map results back
            for idx, farm in farm_centroids_df.iterrows():
                lon, lat = farm['longitude'], farm['latitude']
                try:
                    row, col = rasterio.transform.rowcol(src.transform, lon, lat)
                except Exception as e:
                    print(f"Could not convert lon/lat to pixel: {e}")
                    continue
                window, (left, top) = center_crop_window(row, col, img_width, img_height, window_size)
                crop_img = src.read(window=window)
                if crop_img.shape[0] > 3:
                    crop_img = crop_img[:3]
                elif crop_img.shape[0] < 3:
                    crop_img = np.tile(crop_img, (3 // crop_img.shape[0], 1, 1))
                pil_crop = Image.fromarray(np.moveaxis(crop_img, 0, -1))
                with torch.inference_mode():
                    results_crop = model([pil_crop], device=device, batch=1, verbose=False)[0]
                results_crop = maximum_likelihood_suppression(results_crop)
                offset_xy = (left, top)
                crop_object_info, _ = overlay_segmentation_on_array(
                    results_crop, src.window_transform(window), None,
                    f"{os.path.basename(tif_image)}_farm{idx}", offset_xy)
                all_cropped_object_infos.extend(crop_object_info)

    pd.DataFrame(all_cropped_object_infos).to_csv(final_csv_path, index=False)
    print(f"✅ Targeted centroid-based results saved to {final_csv_path}")


# ======================== Full Pipeline Runner ========================
def run_inference_centroid_targeted(
    folder_path, model_path, final_csv_path, window_size=3000, batch_size=16,
    eps_house=50, min_samples=2, eps_merge=100
):
    model = YOLO(model_path).to('cuda' if torch.cuda.is_available() else 'cpu')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    print(f"Using device: {device}")

    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    all_cropped_object_infos = []

    for tif_image in tqdm(tif_images, desc="GeoTIFFs"):
        # 1. Run initial sliding window inference to get all object_infos for farm detection
        slices = slide_geotiff(tif_image, window_size, window_size)  # stride=window_size for full tiles
        initial_object_infos = []
        for batch in [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]:
            pil_imgs_batch = [Image.fromarray(np.moveaxis(s["img_array"][:3], 0, -1)) for s in batch]
            with torch.inference_mode():
                results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)
            for slice_data, results in zip(batch, results_batch):
                results = maximum_likelihood_suppression(results)
                object_info, _ = overlay_segmentation_on_array(
                    results, slice_data["transform"], None, os.path.basename(tif_image), slice_data["offset_xy"])
                initial_object_infos.extend(object_info)

        # 2. Cluster into farms using centroids
        farm_centroids_df = detect_poultry_farms(initial_object_infos, eps_house, min_samples, eps_merge)
        if farm_centroids_df.empty:
            continue

        # 3. For each farm centroid, crop a window, run inference, map back to image coordinates
        with rasterio.open(tif_image) as src:
            for idx, farm in farm_centroids_df.iterrows():
                lon, lat = farm['longitude'], farm['latitude']
                # Convert lon/lat to pixel
                try:
                    row, col = rasterio.transform.rowcol(src.transform, lon, lat)
                except Exception as e:
                    print(f"⚠️ Could not locate farm centroid in image: {e}")
                    continue

                # Center window on (row, col)
                half_w = window_size // 2
                win_col_off = max(col - half_w, 0)
                win_row_off = max(row - half_w, 0)
                win_width = min(window_size, src.width - win_col_off)
                win_height = min(window_size, src.height - win_row_off)
                window = Window(win_col_off, win_row_off, win_width, win_height)
                crop_img = src.read(window=window)
                crop_transform = src.window_transform(window)

                # Run inference on the cropped image
                pil_crop = Image.fromarray(np.moveaxis(crop_img[:3], 0, -1))
                with torch.inference_mode():
                    results_crop = model([pil_crop], device=device, batch=1, verbose=False)[0]
                results_crop = maximum_likelihood_suppression(results_crop)

                # Use the farm index/id as the tile_name for tracking (optional)
                offset_xy = (win_col_off, win_row_off)
                crop_object_info, _ = overlay_segmentation_on_array(
                    results_crop, crop_transform, None,
                    f"{os.path.basename(tif_image)}_farm{idx}", offset_xy)

                # (a) Get YOLO object info
                crop_object_info, _ = overlay_segmentation_on_array(
                    results_crop, crop_transform, None,
                    f"{os.path.basename(tif_image)}_farm{idx}", offset_xy)
                
                # (b) Run SAM on each YOLO min rectangle
                sam_infos = run_sam_on_min_rects(
                    crop_img, crop_transform, crop_object_info, offset_xy, predictor
                )
                all_cropped_object_infos.extend(sam_infos)

    # Save results
    pd.DataFrame(all_cropped_object_infos).to_csv(final_csv_path, index=False)
    print(f"✅ Targeted centroid-based results saved to {final_csv_path}")



In [1]:
import os
import glob
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from ultralytics import YOLO
import torch
import rasterio
from rasterio.windows import Window
from shapely.geometry import Polygon
from sklearn.cluster import DBSCAN
import cv2

# ---- SAM import and setup ----
from segment_anything import sam_model_registry, SamPredictor

# Setup SAM model (make sure checkpoint path is correct)
sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"
device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device)
predictor = SamPredictor(sam)

# ========== Helper Functions ==========

def maximum_likelihood_suppression(results, conf_threshold=0.7, iou_threshold=0.5):
    from torchvision.ops import nms
    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return results
    boxes = results.boxes.xyxy
    scores = results.boxes.conf
    keep = scores >= conf_threshold
    if keep.sum() == 0:
        results.boxes.data = results.boxes.data[[]]
        if results.masks and results.masks.data is not None:
            results.masks.data = results.masks.data[[]]
        return results
    keep_indices = nms(boxes[keep], scores[keep], iou_threshold)
    indices_to_keep = torch.nonzero(keep).squeeze(1)[keep_indices]
    results.boxes.data = results.boxes.data[indices_to_keep]
    if results.masks and results.masks.data is not None:
        results.masks.data = results.masks.data[indices_to_keep]
    return results

def overlay_segmentation_on_array(results, transform, img_shape, tif_image, offset_xy):
    object_data = []
    masks_out = []

    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return object_data, masks_out

    for idx, (mask, box, cls, conf) in enumerate(zip(
        results.masks.xy, results.boxes.xyxy, results.boxes.cls, results.boxes.conf)):

        cls = int(cls.item())
        conf = float(conf.item())
        x1, y1, x2, y2 = map(float, box.tolist())

        mask_pixel_coords = np.array(mask.tolist(), dtype=np.float32)
        mask_gis_coords = []
        for px, py in mask_pixel_coords:
            try:
                lon, lat = rasterio.transform.xy(transform, py, px)
                mask_gis_coords.append([lon, lat])
            except:
                continue

        pixel_count = len(mask_pixel_coords)
        area = pixel_count * (transform.a * abs(transform.e))
        local_cx = (x1 + x2) / 2
        local_cy = (y1 + y2) / 2
        global_x = offset_xy[0] + local_cx
        global_y = offset_xy[1] + local_cy
        lon_center, lat_center = rasterio.transform.xy(transform, local_cy, local_cx)
        lon_x1, lat_y1 = rasterio.transform.xy(transform, y1, x1)
        lon_x2, lat_y2 = rasterio.transform.xy(transform, y2, y2)
        bbox_width_geo = abs(lon_x2 - lon_x1)
        bbox_height_geo = abs(lat_y2 - lat_y1)

        object_data.append({
            "latitude": lat_center,
            "longitude": lon_center,
            "bbox_lat_y1": lat_y1,
            "bbox_lon_x1": lon_x1,
            "bbox_lat_y2": lat_y2,
            "bbox_lon_x2": lon_x2,
            "bbox_width_geo": bbox_width_geo,
            "bbox_height_geo": bbox_height_geo,
            "global_x": global_x,
            "global_y": global_y,
            "x_center": local_cx,
            "y_center": local_cy,
            "bbox_x1": x1,
            "bbox_y1": y1,
            "bbox_x2": x2,
            "bbox_y2": y2,
            "bbox_width": x2 - x1,
            "bbox_height": y2 - y1,
            "area_geo": area,
            "pixel_count": pixel_count,
            "class": cls,
            "confidence": conf,
            "tile_name": tif_image,
            "mask_index": idx,
            "mask_points": json.dumps(mask_pixel_coords.tolist()),
            "mask_points_gis": json.dumps(mask_gis_coords)
        })

        masks_out.append(mask_pixel_coords)
    return object_data, masks_out

def detect_poultry_farms(object_infos, eps_house=50, min_samples=2, eps_merge=100):
    filtered_infos = [
        o for o in object_infos
        if 'latitude' in o and 'longitude' in o and pd.notnull(o['latitude']) and pd.notnull(o['longitude'])
    ]
    if not filtered_infos:
        return pd.DataFrame(columns=['merged_farm_id', 'longitude', 'latitude', 'num_houses'])
    df = pd.DataFrame(filtered_infos)
    coords = df[['longitude', 'latitude']].values
    dbscan = DBSCAN(eps=eps_house, min_samples=min_samples).fit(coords)
    df['initial_cluster'] = dbscan.labels_
    clustered = df[df['initial_cluster'] != -1].copy()
    if len(clustered) == 0:
        return pd.DataFrame(columns=['merged_farm_id', 'longitude', 'latitude', 'num_houses'])
    centroids = clustered.groupby('initial_cluster')[['longitude', 'latitude']].mean().reset_index()
    centroid_coords = centroids[['longitude', 'latitude']].values
    merge_cluster = DBSCAN(eps=eps_merge, min_samples=1).fit(centroid_coords)
    centroids['merged_farm_id'] = merge_cluster.labels_
    cluster_map = dict(zip(centroids['initial_cluster'], centroids['merged_farm_id']))
    clustered['merged_farm_id'] = clustered['initial_cluster'].map(cluster_map)
    farms = clustered.groupby('merged_farm_id')[['longitude', 'latitude']].mean().reset_index()
    farms['num_houses'] = clustered.groupby('merged_farm_id').size().values
    return farms

def slide_geotiff(geotiff_path, window_size, stride):
    from rasterio.windows import Window
    from rasterio.transform import from_origin
    tiles = []
    with rasterio.open(geotiff_path) as src:
        height, width = src.height, src.width
        x_starts = [0] if width <= window_size else list(range(0, width - window_size + 1, stride))
        if x_starts[-1] + window_size < width:
            x_starts.append(width - window_size)
        y_starts = [0] if height <= window_size else list(range(0, height - window_size + 1, stride))
        if y_starts[-1] + window_size < height:
            y_starts.append(height - window_size)
        for y in y_starts:
            for x in x_starts:
                window = Window(x, y, window_size, window_size)
                slice_img = src.read(window=window)
                new_transform = from_origin(
                    src.transform.c + x * src.transform.a,
                    src.transform.f + y * src.transform.e,
                    src.transform.a,
                    -src.transform.e
                )
                tiles.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "crs": src.crs,
                    "position": (y, x),
                    "offset_xy": (x, y)
                })
    return tiles

def run_sam_on_min_rects(crop_img, crop_transform, yolo_object_infos, offset_xy, predictor):
    """
    For each YOLO object: get min bounding rectangle of mask, crop region, run SAM, save results.
    """
    sam_objects = []
    img_rgb = np.moveaxis(crop_img[:3], 0, -1)
    img_rgb = np.clip((img_rgb / img_rgb.max()) * 255, 0, 255).astype(np.uint8)
    predictor.set_image(img_rgb)
    for obj in yolo_object_infos:
        mask_pts = np.array(json.loads(obj["mask_points"]), dtype=np.int32)
        poly = Polygon(mask_pts)
        if not poly.is_valid or poly.is_empty:
            continue
        min_rect = poly.minimum_rotated_rectangle
        min_rect_coords = np.array(list(min_rect.exterior.coords), dtype=np.int32)
        center_x = int(min_rect.centroid.x)
        center_y = int(min_rect.centroid.y)
        input_point = np.array([[center_x, center_y]])
        input_label = np.array([1])
        masks, scores, logits = predictor.predict(
            point_coords=input_point,
            point_labels=input_label,
            multimask_output=False
        )
        mask = masks[0]
        mask_points = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
        if len(mask_points) == 0:
            continue
        mask_points = mask_points[0].squeeze()
        if mask_points.ndim != 2 or mask_points.shape[0] < 3:
            continue
        mask_gis_coords = []
        for px, py in mask_points:
            try:
                lon, lat = rasterio.transform.xy(crop_transform, py, px)
                mask_gis_coords.append([lon, lat])
            except:
                continue
        lon_center, lat_center = rasterio.transform.xy(crop_transform, center_y, center_x)
        pixel_count = mask.sum()
        area_geo = pixel_count * (crop_transform.a * abs(crop_transform.e))
        sam_objects.append({
            **obj,
            "sam_mask_points": json.dumps(mask_points.tolist()),
            "sam_mask_points_gis": json.dumps(mask_gis_coords),
            "sam_center_lat": lat_center,
            "sam_center_lon": lon_center,
            "sam_area_geo": area_geo,
            "sam_pixel_count": int(pixel_count)
        })
    return sam_objects

# ========== Main Pipeline ==========

def run_inference_centroid_targeted(
    folder_path, model_path, final_csv_path, window_size=3000, batch_size=16,
    eps_house=50, min_samples=2, eps_merge=100
):
    model = YOLO(model_path).to('cuda' if torch.cuda.is_available() else 'cpu')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    print(f"Using device: {device}")

    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    all_cropped_object_infos = []

    for tif_image in tqdm(tif_images, desc="GeoTIFFs"):
        # 1. Run initial sliding window inference to get all object_infos for farm detection
        slices = slide_geotiff(tif_image, window_size, window_size)  # stride=window_size for full tiles
        initial_object_infos = []
        for batch in [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]:
            pil_imgs_batch = [Image.fromarray(np.moveaxis(s["img_array"][:3], 0, -1)) for s in batch]
            with torch.inference_mode():
                results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)
            for slice_data, results in zip(batch, results_batch):
                results = maximum_likelihood_suppression(results)
                object_info, _ = overlay_segmentation_on_array(
                    results, slice_data["transform"], None, os.path.basename(tif_image), slice_data["offset_xy"])
                initial_object_infos.extend(object_info)

        # 2. Cluster into farms using centroids
        farm_centroids_df = detect_poultry_farms(initial_object_infos, eps_house, min_samples, eps_merge)
        if farm_centroids_df.empty:
            continue

        # 3. For each farm centroid, crop a window, run inference, map back to image coordinates
        with rasterio.open(tif_image) as src:
            for idx, farm in farm_centroids_df.iterrows():
                lon, lat = farm['longitude'], farm['latitude']
                try:
                    row, col = rasterio.transform.rowcol(src.transform, lon, lat)
                except Exception as e:
                    print(f"⚠️ Could not locate farm centroid in image: {e}")
                    continue
                half_w = window_size // 2
                win_col_off = max(col - half_w, 0)
                win_row_off = max(row - half_w, 0)
                win_width = min(window_size, src.width - win_col_off)
                win_height = min(window_size, src.height - win_row_off)
                window = Window(win_col_off, win_row_off, win_width, win_height)
                crop_img = src.read(window=window)
                crop_transform = src.window_transform(window)
                pil_crop = Image.fromarray(np.moveaxis(crop_img[:3], 0, -1))
                with torch.inference_mode():
                    results_crop = model([pil_crop], device=device, batch=1, verbose=False)[0]
                results_crop = maximum_likelihood_suppression(results_crop)
                offset_xy = (win_col_off, win_row_off)
                crop_object_info, _ = overlay_segmentation_on_array(
                    results_crop, crop_transform, None,
                    f"{os.path.basename(tif_image)}", offset_xy
                )
                sam_infos = run_sam_on_min_rects(
                    crop_img, crop_transform, crop_object_info, offset_xy, predictor
                )
                all_cropped_object_infos.extend(sam_infos)

    pd.DataFrame(all_cropped_object_infos).to_csv(final_csv_path, index=False)
    print(f"✅ Targeted centroid-based SAM+YOLO results saved to {final_csv_path}")

# ========== Example Usage ==========

# run_inference_centroid_targeted(
#     folder_path="PATH/TO/GEOTIFFS",
#     model_path="PATH/TO/YOUR/YOLO/MODEL.pt",
#     final_csv_path="results_sam_yolo.csv",
#     window_size=3000,
#     batch_size=8
# )


In [2]:
# # folder_path="D:/Data/al_30cm_2021_transfer_1_of_2/31085",
# # folder_path="C:/Users/mza0288/Documents/Code/geotiff_sample",
# folder_path="C:/Users/mza0288/Documents/Code/test_sample",
# folder_path="D:/Data/"
run_inference_centroid_targeted(
    folder_path="C:/Users/mza0288/Documents/Code/test_sample",
    model_path="C:/Users/mza0288/Documents/CODE/runs/segment/yolo11s-500-sgd-04082025/weights/best.pt",
    final_csv_path="sam_test/results_sam_yolo.csv",
    window_size=3000,
    batch_size=8
)

Using device: cuda


GeoTIFFs: 100%|████████████████████████████████████████████████████████████████████████| 10/10 [06:52<00:00, 41.28s/it]

✅ Targeted centroid-based SAM+YOLO results saved to sam_test/results_sam_yolo.csv


In [6]:
import os
import glob
import cv2
import json
import numpy as np
import pandas as pd
import rasterio
from tqdm import tqdm

def draw_sam_annotations_on_image(full_image_array, object_infos, output_path):
    img = np.moveaxis(full_image_array[:3], 0, -1)
    img = np.clip((img / img.max()) * 255, 0, 255).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    overlay = img.copy()

    for obj in object_infos:
        # Bounding box
        x1, y1, x2, y2 = int(obj['bbox_x1']), int(obj['bbox_y1']), int(obj['bbox_x2']), int(obj['bbox_y2'])
        cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Draw SAM mask (no offset addition!)
        try:
            mask_points = np.array(json.loads(obj['sam_mask_points']), dtype=np.int32)
            if len(mask_points.shape) == 2 and mask_points.shape[0] >= 3:
                mask_img = np.zeros_like(img)
                cv2.fillPoly(mask_img, [mask_points], (0, 0, 255))
                cv2.polylines(mask_img, [mask_points], isClosed=True, color=(0, 0, 255), thickness=2)
                overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.4, 0)
        except Exception as e:
            print(f"Failed to draw SAM mask: {e}")

        # Draw class/conf label
        try:
            label = f"Cls:{obj['class']} ({obj['confidence']:.2f})"
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 1.2
            thickness = 3
            text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]
            text_origin = (x1, max(y1 - 10, text_size[1]))
            cv2.putText(overlay, label, text_origin, font, font_scale, (0, 0, 0), thickness + 2)
            cv2.putText(overlay, label, text_origin, font, font_scale, (255, 255, 255), thickness)
        except:
            pass

        # Draw centroid
        try:
            cx, cy = int(obj['x_center']), int(obj['y_center'])
            cv2.circle(overlay, (cx, cy), 6, (0, 0, 255), -1)
        except:
            pass

    cv2.imwrite(output_path, overlay)
    print(f"📸 Annotated image saved to: {output_path}")



def run_viz_sam(folder_path, csv_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    suppressed_infos = pd.read_csv(csv_path)
    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)

    # Normalize tile_name if CSV has additional postfix
    def clean_tile_name(s):
        if isinstance(s, str):
            return s.split(',')[0]
        return s

    suppressed_infos['tile_name_clean'] = suppressed_infos['tile_name'].apply(clean_tile_name)

    for tif_image in tqdm(tif_images, desc="Visualizing SAM overlays"):
        image_name = os.path.basename(tif_image)
        with rasterio.open(tif_image) as src:
            full_image_array = src.read()

        matched_objs = suppressed_infos[suppressed_infos['tile_name_clean'] == image_name]
        matched_objs = matched_objs.to_dict(orient='records')

        if len(matched_objs) == 0:
            print(f"⚠️ No objects for {image_name}")
            continue

        outpath = os.path.join(output_dir, f"{os.path.splitext(image_name)[0]}_sam_annotated.jpg")
        draw_sam_annotations_on_image(full_image_array, matched_objs, outpath)


In [7]:
# --- USAGE EXAMPLE ---
run_viz_sam(
    folder_path="C:/Users/mza0288/Documents/Code/test_sample",
    csv_path="sam_test/results_sam_yolo.csv",
    output_dir="sam_test"
)

Visualizing SAM overlays:  10%|█████▋                                                   | 1/10 [01:27<13:06, 87.38s/it]

📸 Annotated image saved to: sam_test\m_3108501_ne_16_030_20211103_sam_annotated.jpg


Visualizing SAM overlays:  20%|███████████▍                                             | 2/10 [02:15<08:35, 64.48s/it]

📸 Annotated image saved to: sam_test\m_3108501_nw_16_030_20211103_sam_annotated.jpg


Visualizing SAM overlays:  30%|█████████████████                                        | 3/10 [03:59<09:35, 82.16s/it]

...

📸 Annotated image saved to: sam_test\m_3108503_se_16_030_20211103_sam_annotated.jpg


Visualizing SAM overlays:  90%|███████████████████████████████████████████████████▎     | 9/10 [09:20<00:53, 54.00s/it]

📸 Annotated image saved to: sam_test\m_3108503_sw_16_030_20211103_sam_annotated.jpg


Visualizing SAM overlays: 100%|████████████████████████████████████████████████████████| 10/10 [10:24<00:00, 62.49s/it]

📸 Annotated image saved to: sam_test\m_3108504_sw_16_030_20211114_sam_annotated.jpg
